#### Audio and Speech
- 음성 비서
     - AI 콜센터, 음성 챗봇, 자동차 음성 비서 등
     - speech-to-speech(Realtime)
- Text-to-Speech (텍스트를 사람 목소리로 변환)
- Speech-toText (음성을 텍스트로 변환)
- Realtime API (audio, text => audio, text), Transcription API(audio => text), Speech API(text => audio) 사용,
  chat.completions(audio, text => audio, text)
- Response API 는 지원안함

In [1]:
from dotenv import load_dotenv, find_dotenv
# .env 파일 정보 가져오기
load_dotenv(find_dotenv())

True

In [2]:
import base64
from openai import OpenAI

client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-audio",
    modalities=["text", "audio"],
    audio={"voice": "alloy", "format": "wav"},
    messages=[
        {
            "role": "user",
            "content": "Is a golden retriever a good family dog?"
        }
    ]
)

print(completion.choices[0])

wav_bytes = base64.b64decode(completion.choices[0].message.audio.data)
with open("./audio/dog.wav", "wb") as f:
    f.write(wav_bytes)

Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=ChatCompletionAudio(id='audio_69981514ecbc819187f24b1938cc7f30', data='...', expires_at=1771578148, transcript='Absolutely! Golden retrievers are generally considered fantastic family dogs. They are known for their friendly and gentle temperament. They get along well with children, are eager to please, and are highly trainable. They do need regular exercise and attention, but their loving and loyal nature makes them a great choice for many families.'), function_call=None, tool_calls=None))


In [5]:
# text => 음성
speech = client.audio.speech.create(
    model="gpt-4o-mini-tts",
    voice="ash",
    input="안녕하세요"
)
with open("./audio/tts.wav", "wb") as f:
    f.write(speech.read())

In [7]:
# 음성 => 텍스트
with open("./audio/obama.mp3", "rb") as f:
    transcript = client.audio.transcriptions.create(
        model="gpt-4o-transcribe",
        file=f
    )
print(transcript.text)

Hello, Chicago! It's good to be home! Tonight, it's my turn to say thanks. And every day I have learned from you. You made me a better president and you made me a better man. I can't do that. I learned that change only happens when ordinary people get involved and they get engaged and they come together to demand it. After eight years as your president, I still believe that. In ten days, the world will witness a hallmark of our democracy. No, no, no, no, no. The peaceful transfer of power from one freely elected president to the next. I committed to President-elect Trump that my administration would ensure the smoothest possible transition, just as President Bush did for me. After all, we remain the wealthiest, most powerful, and most respected nation on earth. Michelle, for the past 25 years you have not only been my wife and mother of my children, you have been my best friend. You have made me proud and you have made the country proud. My fellow Americans, it has been the honor of my

In [8]:
import base64
import requests
from openai import OpenAI

client = OpenAI()

# Fetch the audio file and convert it to a base64 encoded string
url = "https://cdn.openai.com/API/docs/audio/alloy.wav"
response = requests.get(url)

# response 애러발생
response.raise_for_status()
# byte 코드로 가져오기
wav_data = response.content

encoded_string = base64.b64encode(wav_data).decode('utf-8')

completion = client.chat.completions.create(
    model="gpt-audio",
    modalities=["text", "audio"],
    audio={"voice": "alloy", "format": "wav"},
    messages=[
        {
            "role": "user",
            "content": [
                { 
                    "type": "text",
                    "text": "What is in this recording?"
                },
                {
                    "type": "input_audio",
                    "input_audio": {
                        "data": encoded_string,
                        "format": "wav"
                    }
                }
            ]
        },
    ]
)

print(completion.choices[0].message)

# 답변으로 받은 텍스트출력
print(f"답변 {completion.choices[0].message.audio.transcript}")

# 답변으로 받은 오디오 출력
audio_base64 = completion.choices[0].message.audio.data
audio_bytes = base64.b64decode(audio_base64)

with open("./audio/response.wav", "wb") as fw:
    fw.write(audio_bytes)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=ChatCompletionAudio(id='audio_69981e2ac7f881919ac7637319a04cbe', data='...', expires_at=1771580474, transcript="The recording contains a statement about the movement of the sun, describing how it rises in the east and sets in the west, and mentions that this has been observed by humans for thousands of years. There isn't any music or background noise—just a factual description of a natural phenomenon."), function_call=None, tool_calls=None)
답변 The recording contains a statement about the movement of the sun, describing how it rises in the east and sets in the west, and mentions that this has been observed by humans for thousands of years. There isn't any music or background noise—just a factual description of a natural phenomenon.
